<a href="https://colab.research.google.com/github/Davron030901/Machine_Learning/blob/main/m2_c4_vectors_distances.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Math Foundations Part 1 -- Vectors and Distances
## Module 2, Class 4

**Objective:** Understand vectors, distances, and dot products -- the building blocks behind ML algorithms.

### What you will practice
- Representing real data as vectors
- Computing Euclidean distance by hand and with NumPy
- Dot product: manual and with np.dot()
- Linear combinations for scoring

---

## 0. Setup

In [1]:
import numpy as np
import pandas as pd

print("Setup complete.")

Setup complete.


## 1. Products as Vectors

Every ML algorithm sees data as numbers in space. A product with Price, Quantity, and Discount is a point in 3D space.

We pick 3 products from the Superstore dataset:

In [2]:
# 3 products: [Price, Quantity, Discount]
product_a = np.array([261.96, 2, 0.00])   # Office Supplies order
product_b = np.array([731.94, 3, 0.00])   # Technology order
product_c = np.array([14.62,  6, 0.20])   # Small discounted order

products = pd.DataFrame(
    [product_a, product_b, product_c],
    columns=['Price', 'Quantity', 'Discount'],
    index=['Product A', 'Product B', 'Product C']
)
products

,Price,Quantity,Discount
Product A,261.96,2.0,0.0
Product B,731.94,3.0,0.0
Product C,14.62,6.0,0.2


## 2. Euclidean Distance

Euclidean distance between two vectors $\mathbf{a}$ and $\mathbf{b}$ in $n$ dimensions:

$$d(\mathbf{a}, \mathbf{b}) = \sqrt{\sum_{i=1}^{n}(a_i - b_i)^2}$$

Let's compute the distance between Product A and Product B **by hand** first.

In [3]:
# Manual Euclidean distance: A to B
diff = product_a - product_b
print(f"Difference vector: {diff}")

squared = diff ** 2
print(f"Squared differences: {squared}")

sum_squared = np.sum(squared)
print(f"Sum of squares: {sum_squared}")

dist_manual = np.sqrt(sum_squared)
print(f"\nEuclidean distance (A to B): {dist_manual:.4f}")

Difference vector: [-469.98   -1.      0.  ]
Squared differences: [2.208812e+05 1.000000e+00 0.000000e+00]
Sum of squares: 220882.20040000006

Euclidean distance (A to B): 469.9811


In [4]:
# Verify with NumPy
dist_numpy = np.linalg.norm(product_a - product_b)
print(f"np.linalg.norm distance (A to B): {dist_numpy:.4f}")
print(f"Match: {np.isclose(dist_manual, dist_numpy)}")

np.linalg.norm distance (A to B): 469.9811
Match: True


In [5]:
# Distance matrix: all pairs
pairs = [('A-B', product_a, product_b),
         ('A-C', product_a, product_c),
         ('B-C', product_b, product_c)]

for name, v1, v2 in pairs:
    d = np.linalg.norm(v1 - v2)
    print(f"Distance {name}: {d:.4f}")

print("\nWhich two products are most similar (closest)?")

Distance A-B: 469.9811
Distance A-C: 247.3724
Distance B-C: 717.3263

Which two products are most similar (closest)?


**Note:** Price dominates the distance because its values are much larger than Quantity or Discount. This is exactly why we need **feature scaling** (Module 3, Class 2).

---

## 3. Dot Product

The dot product of two vectors:

$$\mathbf{a} \cdot \mathbf{b} = \sum_{i=1}^{n} a_i \cdot b_i$$

It measures how much two vectors "agree" in direction. Used everywhere: linear regression coefficients, neural network layers, similarity scores.

In [6]:
# Manual dot product: A dot B
elementwise = product_a * product_b
print(f"Element-wise products: {elementwise}")

dot_manual = np.sum(elementwise)
print(f"Sum (dot product): {dot_manual:.4f}")

Element-wise products: [1.91739002e+05 6.00000000e+00 0.00000000e+00]
Sum (dot product): 191745.0024


In [7]:
# Verify with np.dot()
dot_numpy = np.dot(product_a, product_b)
print(f"np.dot(A, B): {dot_numpy:.4f}")
print(f"Match: {np.isclose(dot_manual, dot_numpy)}")

np.dot(A, B): 191745.0024
Match: True


In [8]:
# All dot products
for name, v1, v2 in pairs:
    d = np.dot(v1, v2)
    print(f"Dot product {name}: {d:.4f}")

print("\nHigher dot product = vectors point in more similar directions.")

Dot product A-B: 191745.0024
Dot product A-C: 3841.8552
Dot product B-C: 10718.9628

Higher dot product = vectors point in more similar directions.


---
## 4. Linear Combination: Building a Scoring Function

A linear combination applies weights to features to produce a single score. This is how linear regression works at its core.

$$\text{score} = w_1 \cdot \text{Sales} + w_2 \cdot \text{Quantity} + w_3 \cdot \text{Discount}$$

Weights reflect importance: positive = good, negative = penalizes.

In [9]:
# Define weights: Sales matters most, Quantity helps, Discount hurts
weights = np.array([0.5, 0.3, -0.2])

# Score each product using dot product (that's all a linear combination is)
for name, vec in [('A', product_a), ('B', product_b), ('C', product_c)]:
    score = np.dot(weights, vec)
    print(f"Product {name}: score = 0.5*{vec[0]} + 0.3*{vec[1]} + (-0.2)*{vec[2]} = {score:.2f}")

print("\nThe dot product IS the linear combination. Same operation.")

Product A: score = 0.5*261.96 + 0.3*2.0 + (-0.2)*0.0 = 131.58
Product B: score = 0.5*731.94 + 0.3*3.0 + (-0.2)*0.0 = 366.87
Product C: score = 0.5*14.62 + 0.3*6.0 + (-0.2)*0.2 = 9.07

The dot product IS the linear combination. Same operation.


In [10]:
# Vectorized: score all products at once using matrix multiplication
all_products = np.array([product_a, product_b, product_c])
scores = all_products @ weights  # matrix-vector multiplication

result = pd.DataFrame({
    'Product': ['A', 'B', 'C'],
    'Price': all_products[:, 0],
    'Quantity': all_products[:, 1],
    'Discount': all_products[:, 2],
    'Score': scores
})
result.sort_values('Score', ascending=False)

,Product,Price,Quantity,Discount,Score
1,B,731.94,3.0,0.0,366.87
0,A,261.96,2.0,0.0,131.58
2,C,14.62,6.0,0.2,9.07


**Connection to ML:** When you train a linear regression, the model learns the optimal weights $w_1, w_2, w_3$ automatically. You just saw the math it does under the hood.

---

## TODO: Your Turn

Define 3 more products with different [Price, Quantity, Discount] values. Then:

1. Compute pairwise Euclidean distances (manual + verify with NumPy)
2. Compute dot products between all pairs
3. Score each product with weights `[0.4, 0.4, -0.3]`
4. Which product scores highest? Does it make intuitive sense?

In [11]:
# TODO: Define 3 new products
product_d = np.array([0, 0, 0])  # Replace with real values
product_e = np.array([0, 0, 0])  # Replace with real values
product_f = np.array([0, 0, 0])  # Replace with real values

# TODO: Compute pairwise Euclidean distances
# Your code here


In [12]:
# TODO: Compute dot products between all pairs
# Your code here


In [13]:
# TODO: Score each product with weights [0.4, 0.4, -0.3]
new_weights = np.array([0.4, 0.4, -0.3])
# Your code here


*TODO: Which product scored highest? Does it make intuitive sense given the weights?*

---
## Summary

| Concept | What it does | Where it shows up in ML |
|---------|-------------|------------------------|
| Vector | Represents a data point as numbers | Every row in your dataset |
| Euclidean distance | Measures how far apart two points are | KNN, clustering |
| Dot product | Measures alignment between vectors | Linear regression, neural networks |
| Linear combination | Weighted sum of features | Model predictions |